In [ ]:
import torch
import torch.nn.functional as F
from torch.fft import fft2, ifft2
import torch.nn as nn
from torch import Tensor

from PIL import Image
import numpy as np

import matplotlib.pyplot as plt
import matplotlib

import os
import math
from typing import Tuple, List
# from util.vis import vis_voxelgrid
from util.vis import vis_depth_map

In [ ]:
def psf2otf(psf, shape):
    """
    将点扩散函数(PSF)转换为光学传递函数(OTF)
    Args:
        psf: 输入滤波器，形状为(..., H, W)
        shape: 目标输出尺寸(H, W)
    Returns:
        otf: 频域滤波器，形状为(..., H, W)
    """
    # 旋转180度以匹配MATLAB的psf2otf行为
    rotated_psf = torch.flip(psf, dims=(-2, -1))
    
    # 计算需要填充的尺寸
    h, w = psf.shape[-2], psf.shape[-1]
    target_h, target_w = shape
    pad_h = target_h - h
    pad_w = target_w - w
    
    # 在右下侧填充零（MATLAB默认行为）
    padded_psf = F.pad(rotated_psf, (0, pad_w, 0, pad_h), mode='constant', value=0)
    
    # 计算FFT并返回
    return fft2(padded_psf)

def ssf_filter_batch(I, alpha=0.8, beta=0.05, kappa=1.5, tau=0.8, iter_max=500, lambda_max=1e2):
    """
    半稀疏平滑滤波器（支持批处理版本）
    
    Args:
        I (torch.Tensor): 输入张量，形状为(B, C, H, W)，值域建议[0,1]
        alpha: 一阶梯度平滑权重（默认0.8）
        beta: 二阶梯度阈值松弛因子（默认0.05）
        kappa: 正则项权重增长系数（默认1.5）
        tau: 平滑权重衰减系数（默认0.8）
        iter_max: 最大迭代次数（默认500）
        lambda_max: 最大正则化权重（默认1e2）
        
    Returns:
        torch.Tensor: 滤波结果，形状与输入相同
    """
    device = I.device
    B, C, H, W = I.shape  # 批处理维度支持
    
    # ================== 滤波器定义 ==================
    # 一阶差分滤波器（水平/垂直）
    Dx = torch.tensor([[1.0, -1.0]], device=device) / 2.0  # 水平梯度
    Dy = torch.tensor([[1.0], [-1.0]], device=device) / 2.0  # 垂直梯度
    
    # 二阶差分滤波器
    fxx = torch.tensor([[1.0, -2.0, 1.0]], device=device) / 4.0  # 水平二阶
    fyy = torch.tensor([[1.0], [-2.0], [1.0]], device=device) / 4.0  # 垂直二阶
    fuu = torch.tensor([[1.0, 0.0, 0.0],
                        [0.0, -2.0, 0.0],
                        [0.0, 0.0, 1.0]], device=device) / 4.0  # 对角线1
    fvv = torch.tensor([[0.0, 0.0, 1.0],
                        [0.0, -2.0, 0.0],
                        [1.0, 0.0, 0.0]], device=device) / 4.0  # 对角线2

    # =============== 计算频域分母项 ===============
    # 计算各滤波器的OTF
    otfDx = psf2otf(Dx, (H, W))
    otfDy = psf2otf(Dy, (H, W))
    otfFxx = psf2otf(fxx, (H, W))
    otfFyy = psf2otf(fyy, (H, W))
    otfFuu = psf2otf(fuu, (H, W))
    otfFvv = psf2otf(fvv, (H, W))
    
    # 构造频域分母项
    Denormin1 = torch.abs(otfDx)**2 + torch.abs(otfDy)**2  # 一阶项
    Denormin2 = (torch.abs(otfFxx)**2 + torch.abs(otfFyy)**2 
                + torch.abs(otfFuu)**2 + torch.abs(otfFvv)**2)  # 二阶项
    
    # 扩展维度以支持批处理和通道维度 (B, C, H, W)
    Denormin1 = Denormin1.view(1, 1, H, W).expand(B, C, H, W).to(torch.complex64)
    Denormin2 = Denormin2.view(1, 1, H, W).expand(B, C, H, W).to(torch.complex64)

    # =============== 初始化迭代变量 ===============
    S = I.clone()  # 初始化输出
    Normin0 = fft2(S)  # 输入图像的频域表示
    lambda_val = 10 * beta  # 初始正则化权重
    current_alpha = alpha  # 当前平滑权重

    # =============== 主迭代循环 ===============
    for iter in range(1, iter_max + 1):
        if lambda_val > lambda_max:
            break  # 提前终止

        # 构造当前迭代的分母项
        Denormin = 1.0 + current_alpha * Denormin1 + lambda_val * Denormin2

        # ----------------- 一阶梯度计算 -----------------
        # 水平梯度
        Dx_kernel = Dx.view(1, 1, 1, 2).expand(C, 1, 1, 2).contiguous()  # 批处理扩展
        # (B, C, H, W) --> (B, C, H, W + 1)
        padded_S = F.pad(S, (0, 1, 0, 0), mode='circular')  # 右侧填充1像素
        gx = F.conv2d(padded_S, Dx_kernel, groups=C, padding=0).view(B, C, H, W)
        
        # 垂直梯度
        Dy_kernel = Dy.view(1, 1, 2, 1).expand(C, 1, 2, 1).contiguous()
        # (B, C, H, W) --> (B, C, H + 1, W)
        padded_S = F.pad(S, (0, 0, 0, 1), mode='circular')  # 底部填充1像素
        gy = F.conv2d(padded_S, Dy_kernel, groups=C, padding=0).view(B, C, H, W)

        # ----------------- 二阶梯度计算 -----------------
        # 水平二阶
        fxx_kernel = fxx.view(1, 1, 1, 3).expand(C, 1, 1, 3).contiguous()
        # (B, C, H, W) --> (B, C, H, W + 2)
        padded_S = F.pad(S, (1, 1, 0, 0), mode='circular')  # 水平两侧填充
        gxx = F.conv2d(padded_S, fxx_kernel, groups=C, padding=0).view(B, C, H, W)
        
        # 垂直二阶
        fyy_kernel = fyy.view(1, 1, 3, 1).expand(C, 1, 3, 1).contiguous()
        # (B, C, H, W) --> (B, C, H + 2, W)
        padded_S = F.pad(S, (0, 0, 1, 1), mode='circular')  # 垂直两侧填充
        gyy = F.conv2d(padded_S, fyy_kernel, groups=C, padding=0).view(B, C, H, W)
        
        # 对角线二阶1
        fuu_kernel = fuu.view(1, 1, 3, 3).expand(C, 1, 3, 3).contiguous()
        padded_S = F.pad(S, (1, 1, 1, 1), mode='circular')  # 全方向填充
        guu = F.conv2d(padded_S, fuu_kernel, groups=C, padding=0).view(B, C, H, W)
        
        # 对角线二阶2
        fvv_kernel = fvv.view(1, 1, 3, 3).expand(C, 1, 3, 3).contiguous()
        padded_S = F.pad(S, (1, 1, 1, 1), mode='circular')
        gvv = F.conv2d(padded_S, fvv_kernel, groups=C, padding=0).view(B, C, H, W)

        # ----------------- 联合阈值处理 -----------------
        sum_sq = gxx**2 + gyy**2 + guu**2 + gvv**2  # 各通道梯度平方和
        if C > 1:
            sum_sq = sum_sq.sum(dim=1, keepdim=True)  # 多通道联合求和 (B,1,H,W)
            
        t = sum_sq < (beta / lambda_val)  # 生成联合阈值掩码
        t = t.expand(B, C, H, W)  # 扩展到所有通道
        
        # 应用阈值（置零小梯度）
        gxx = gxx.masked_fill(t, 0.0)
        gyy = gyy.masked_fill(t, 0.0)
        guu = guu.masked_fill(t, 0.0)
        gvv = gvv.masked_fill(t, 0.0)

        # ----------------- 反向投影项计算 -----------------
        # 一阶项的反向投影
        reversed_Dx = torch.flip(Dx_kernel, dims=[-1])
        padded_gx = F.pad(gx, (0, 1, 0, 0), mode='circular')
        Normin_x = F.conv2d(padded_gx, reversed_Dx, groups=C, padding=0).view(B, C, H, W)
        Normin_x = torch.roll(Normin_x, shifts=1, dims=-1)  # 循环移位校正
        
        reversed_Dy = torch.flip(Dy_kernel, dims=[-2])
        padded_gy = F.pad(gy, (0, 0, 0, 1), mode='circular')
        Normin_y = F.conv2d(padded_gy, reversed_Dy, groups=C, padding=0).view(B, C, H, W)
        Normin_y = torch.roll(Normin_y, shifts=1, dims=-2)
        
        Normin1 = Normin_x + Normin_y  # 一阶总项

        # 二阶项的反向投影
        reversed_fxx = torch.flip(fxx_kernel, dims=[-1])
        padded_gxx = F.pad(gxx, (1, 1, 0, 0), mode='circular')
        Normin_xx = F.conv2d(padded_gxx, reversed_fxx, groups=C, padding=0).view(B, C, H, W)
        
        reversed_fyy = torch.flip(fyy_kernel, dims=[-2])
        padded_gyy = F.pad(gyy, (0, 0, 1, 1), mode='circular')
        Normin_yy = F.conv2d(padded_gyy, reversed_fyy, groups=C, padding=0).view(B, C, H, W)
        
        reversed_fuu = torch.flip(fuu_kernel, dims=(-2, -1))
        padded_guu = F.pad(guu, (1, 1, 1, 1), mode='circular')
        Normin_uu = F.conv2d(padded_guu, reversed_fuu, groups=C, padding=0).view(B, C, H, W)
        
        reversed_fvv = torch.flip(fvv_kernel, dims=(-2, -1))
        padded_gvv = F.pad(gvv, (1, 1, 1, 1), mode='circular')
        Normin_vv = F.conv2d(padded_gvv, reversed_fvv, groups=C, padding=0).view(B, C, H, W)
        
        Normin2 = Normin_xx + Normin_yy + Normin_uu + Normin_vv  # 二阶总项

        # ----------------- 频域更新 -----------------
        FS = (Normin0 + current_alpha * fft2(Normin1) + lambda_val * fft2(Normin2)) / Denormin
        S = torch.real(ifft2(FS))  # 回到空间域
        
        # ----------------- 参数更新 -----------------
        current_alpha *= tau  # 衰减平滑权重
        lambda_val *= kappa  # 增强稀疏约束

        # 进度打印
        if iter % 50 == 0:
            print(f'Iteration {iter}, lambda: {lambda_val:.1f}')

    return S.clamp(0, 1)  # 确保输出在[0,1]范围


In [ ]:
def psf2otf(psf, shape):
    rotated_psf = torch.flip(psf, dims=(-2, -1))
    h, w = psf.shape[-2], psf.shape[-1]
    target_h, target_w = shape
    pad_h = target_h - h
    pad_w = target_w - w
    padded_psf = F.pad(rotated_psf, (0, pad_w, 0, pad_h), mode='constant', value=0)
    return torch.fft.fft2(padded_psf)

def ssf_filter_batch(I, alpha=0.8, beta=0.05, kappa=1.5, tau=1.0, iter_max=500, lambda_max=1e2):
    device = I.device
    B, C, H, W = I.shape
    
    # ================== 滤波器定义 ==================
    # 一阶差分滤波器
    Dx = torch.tensor([[1.0, -1.0]], device=device) / 2.0
    Dy = torch.tensor([[1.0], [-1.0]], device=device) / 2.0
    
    # 二阶差分滤波器
    fxx = torch.tensor([[1.0, -2.0, 1.0]], device=device) / 4.0
    fyy = torch.tensor([[1.0], [-2.0], [1.0]], device=device) / 4.0
    fuu = torch.tensor([[1.0, 0.0, 0.0],
                       [0.0, -2.0, 0.0],
                       [0.0, 0.0, 1.0]], device=device) / 4.0
    fvv = torch.tensor([[0.0, 0.0, 1.0],
                       [0.0, -2.0, 0.0],
                       [1.0, 0.0, 0.0]], device=device) / 4.0

    # =============== 预处理输入图像的梯度 ===============
    # 计算输入图像的一阶梯度
    Dx_kernel = Dx.view(1, 1, 1, 2).expand(C, 1, 1, 2).contiguous()
    Dy_kernel = Dy.view(1, 1, 2, 1).expand(C, 1, 2, 1).contiguous()
    
    # 计算Dx_I
    padded_I = F.pad(I, (0, 1, 0, 0), mode='circular')
    Dx_I = F.conv2d(padded_I, Dx_kernel, groups=C, padding=0).view(B, C, H, W)
    
    # 计算Dy_I
    padded_I = F.pad(I, (0, 0, 0, 1), mode='circular')
    Dy_I = F.conv2d(padded_I, Dy_kernel, groups=C, padding=0).view(B, C, H, W)

    # =============== 计算频域分母项 ===============
    otfDx = psf2otf(Dx, (H, W))
    otfDy = psf2otf(Dy, (H, W))
    otfFxx = psf2otf(fxx, (H, W))
    otfFyy = psf2otf(fyy, (H, W))
    otfFuu = psf2otf(fuu, (H, W))
    otfFvv = psf2otf(fvv, (H, W))
    
    Denormin1 = torch.abs(otfDx)**2 + torch.abs(otfDy)**2
    Denormin2 = (torch.abs(otfFxx)**2 + torch.abs(otfFyy)**2 
                + torch.abs(otfFuu)**2 + torch.abs(otfFvv)**2)
    
    Denormin1 = Denormin1.view(1, 1, H, W).expand(B, C, H, W).to(torch.complex64)
    Denormin2 = Denormin2.view(1, 1, H, W).expand(B, C, H, W).to(torch.complex64)

    # =============== 初始化迭代变量 ===============
    S = I.clone()
    Normin0 = torch.fft.fft2(S)
    lambda_val = 10 * beta  # 初始lambda值
    current_alpha = alpha

    # =============== 主迭代循环 ===============
    for iter in range(1, iter_max + 1):
        if lambda_val > lambda_max:
            break

        Denormin = 1.0 + current_alpha * Denormin1 + lambda_val * Denormin2

        # ----------------- 一阶梯度计算 -----------------
        # 计算当前S的一阶梯度
        padded_S = F.pad(S, (0, 1, 0, 0), mode='circular')
        gx = F.conv2d(padded_S, Dx_kernel, groups=C, padding=0).view(B, C, H, W)
        padded_S = F.pad(S, (0, 0, 0, 1), mode='circular')
        gy = F.conv2d(padded_S, Dy_kernel, groups=C, padding=0).view(B, C, H, W)
        
        # 计算梯度差异
        gx_diff = gx - Dx_I
        gy_diff = gy - Dy_I

        # ----------------- 二阶梯度计算 -----------------
        fxx_kernel = fxx.view(1, 1, 1, 3).expand(C, 1, 1, 3).contiguous()
        padded_S = F.pad(S, (1, 1, 0, 0), mode='circular')
        gxx = F.conv2d(padded_S, fxx_kernel, groups=C, padding=0).view(B, C, H, W)
        
        fyy_kernel = fyy.view(1, 1, 3, 1).expand(C, 1, 3, 1).contiguous()
        padded_S = F.pad(S, (0, 0, 1, 1), mode='circular')
        gyy = F.conv2d(padded_S, fyy_kernel, groups=C, padding=0).view(B, C, H, W)
        
        fuu_kernel = fuu.view(1, 1, 3, 3).expand(C, 1, 3, 3).contiguous()
        padded_S = F.pad(S, (1, 1, 1, 1), mode='circular')
        guu = F.conv2d(padded_S, fuu_kernel, groups=C, padding=0).view(B, C, H, W)
        
        fvv_kernel = fvv.view(1, 1, 3, 3).expand(C, 1, 3, 3).contiguous()
        padded_S = F.pad(S, (1, 1, 1, 1), mode='circular')
        gvv = F.conv2d(padded_S, fvv_kernel, groups=C, padding=0).view(B, C, H, W)

        # ----------------- 联合阈值处理 -----------------
        sum_sq = gxx**2 + gyy**2 + guu**2 + gvv**2
        if C > 1:
            sum_sq = sum_sq.sum(dim=1, keepdim=True)
            
        t = sum_sq < (beta / lambda_val)
        t = t.expand(B, C, H, W)
        
        gxx = gxx.masked_fill(t, 0.0)
        gyy = gyy.masked_fill(t, 0.0)
        guu = guu.masked_fill(t, 0.0)
        gvv = gvv.masked_fill(t, 0.0)

        # ----------------- 反向投影项计算 -----------------
        # 一阶差异项的反向投影
        reversed_Dx = torch.flip(Dx_kernel, dims=[-1])
        padded_gx_diff = F.pad(gx_diff, (0, 1, 0, 0), mode='circular')
        Normin_x = F.conv2d(padded_gx_diff, reversed_Dx, groups=C, padding=0).view(B, C, H, W)
        Normin_x = torch.roll(Normin_x, shifts=1, dims=-1)
        
        reversed_Dy = torch.flip(Dy_kernel, dims=[-2])
        padded_gy_diff = F.pad(gy_diff, (0, 0, 0, 1), mode='circular')
        Normin_y = F.conv2d(padded_gy_diff, reversed_Dy, groups=C, padding=0).view(B, C, H, W)
        Normin_y = torch.roll(Normin_y, shifts=1, dims=-2)
        
        Normin1 = Normin_x + Normin_y

        # 二阶项的反向投影
        reversed_fxx = torch.flip(fxx_kernel, dims=[-1])
        padded_gxx = F.pad(gxx, (1, 1, 0, 0), mode='circular')
        Normin_xx = F.conv2d(padded_gxx, reversed_fxx, groups=C, padding=0).view(B, C, H, W)
        
        reversed_fyy = torch.flip(fyy_kernel, dims=[-2])
        padded_gyy = F.pad(gyy, (0, 0, 1, 1), mode='circular')
        Normin_yy = F.conv2d(padded_gyy, reversed_fyy, groups=C, padding=0).view(B, C, H, W)
        
        reversed_fuu = torch.flip(fuu_kernel, dims=(-2, -1))
        padded_guu = F.pad(guu, (1, 1, 1, 1), mode='circular')
        Normin_uu = F.conv2d(padded_guu, reversed_fuu, groups=C, padding=0).view(B, C, H, W)
        
        reversed_fvv = torch.flip(fvv_kernel, dims=(-2, -1))
        padded_gvv = F.pad(gvv, (1, 1, 1, 1), mode='circular')
        Normin_vv = F.conv2d(padded_gvv, reversed_fvv, groups=C, padding=0).view(B, C, H, W)
        
        Normin2 = Normin_xx + Normin_yy + Normin_uu + Normin_vv

        # ----------------- 频域更新 -----------------
        FS = (Normin0 + current_alpha * torch.fft.fft2(Normin1) + lambda_val * torch.fft.fft2(Normin2)) / Denormin
        S = torch.real(torch.fft.ifft2(FS))
        
        # ----------------- 参数更新 -----------------
        # current_alpha *= tau  # 保持alpha不变
        lambda_val *= kappa

        if iter % 50 == 0:
            print(f'Iteration {iter}, lambda: {lambda_val:.1f}')

    return S.clamp(0, 1)

In [ ]:
class InformationRichnessV2(nn.Module):
    def __init__(
        self,
        patch_size: Tuple[int, int] = (14, 14),
        dct_block_size: Tuple[int, int] = (3, 3),
        bins: int = 256,
    ):
        super().__init__()

        self.patch_size = patch_size
        self.dct_block_size = dct_block_size
        self.bins = bins

        # Pre-generate DCT basis matrix (for acceleration)
        self.register_buffer("dct_matrix", self.generate_dct_matrix(dct_block_size[0]))

    def generate_dct_matrix(self, size: int) -> Tensor:
        """Generate an orthogonal DCT-II basis matrix."""
        matrix = torch.zeros(size, size)
        for i in range(size):
            for j in range(size):
                if i == 0:
                    matrix[i, j] = math.sqrt(1 / size)
                else:
                    matrix[i, j] = math.sqrt(2 / size) * math.cos(
                        math.pi * (2 * j + 1) * i / (2 * size)
                    )
        return matrix

    def dct2d(self, blocks: Tensor) -> Tensor:
        """Batch 2D DCT transformation."""
        return torch.matmul(self.dct_matrix, torch.matmul(blocks, self.dct_matrix.T))

    def extract_patches(self, x: Tensor) -> Tensor:
        """Extract patches from image (without padding)."""
        B, C, H, W = x.shape
        ph, pw = self.patch_size
        dh, dw = self.dct_block_size

        # Main patch extraction (14x14 or given patch_size)
        x = x.unfold(2, ph, ph).unfold(3, pw, pw)  # (B, C, H/ph, W/pw, ph, pw)
        x = x.permute(0, 2, 3, 1, 4, 5)  # (B, H_p, W_p, C, ph, pw)

        return x.contiguous()

    def compute_direction_variance(self, dct_blocks: Tensor) -> Tensor:
        """
        Compute the variance of standard deviations across four directions (rows, columns, main diagonal, and anti-diagonal).
        The result is normalized and the variance is calculated.
        """

        """ Calculate standard deviations for each direction """
        # Row-wise standard deviation
        S1 = torch.std(dct_blocks, dim=-1, unbiased=False).mean(dim=-1)

        # Column-wise standard deviation
        S2 = torch.std(dct_blocks, dim=-2, unbiased=False).mean(dim=-1)

        # Main diagonal standard deviation
        diag_main = torch.diagonal(dct_blocks, dim1=-2, dim2=-1)
        S3 = torch.std(diag_main, dim=-1, unbiased=False)

        # Anti-diagonal standard deviation (flip the tensor horizontally and then take the main diagonal)
        flipped = torch.flip(dct_blocks, dims=[-1])
        diag_anti = torch.diagonal(flipped, dim1=-2, dim2=-1)
        S4 = torch.std(diag_anti, dim=-1, unbiased=False)

        """ Normalize the standard deviations to avoid division by zero """
        S = (S1 + S2 + S3 + S4) / 4 + 1e-8
        S1_norm = S1 / S
        S2_norm = S2 / S
        S3_norm = S3 / S
        S4_norm = S4 / S

        """ Calculate the variance of the normalized standard deviations """
        return torch.var(
            torch.stack([S1_norm, S2_norm, S3_norm, S4_norm], dim=-1),
            dim=-1,
            unbiased=True,
        )

    def compute_entropy(self, patches):
        """
        Compute the entropy for each channel of the patches, preserving the channel dimension.
        """
        # Convert pixel values to integers in the range [0, 255]
        patches_int = (patches * 255).round().long().clamp(0, 255)
        patches_flat = patches_int.flatten(start_dim=4)  # [B, H_p, W_p, C, ph*pw]

        # Get tensor dimensions and device
        B, H_p, W_p, C, N = patches_flat.shape
        device = patches_flat.device

        # Expand dimensions for broadcasting comparison
        patches_expanded = patches_flat.unsqueeze(-1)  # [B, H_p, W_p, C, N, 1]
        # Shape: [1, 1, 1, 1, 1, 256]
        values = torch.arange(self.bins, device=device).view(1, 1, 1, 1, 1, self.bins)

        # Count the occurrences of each pixel value
        counts = (patches_expanded == values).sum(dim=4)  # [B, H_p, W_p, C, 256]

        # Compute probabilities for each pixel value
        prob = counts.float() / N + 1e-10
        entropy = -(prob * torch.log2(prob)).sum(dim=-1)  # [B, H_p, W_p, C]
        return entropy

    def forward(self, x: Tensor) -> Tensor:
        """Calculate information richness for each patch."""
        B, C, H, W = x.shape

        # Extract patches
        patches = self.extract_patches(x)  # (B, H_p, W_p, C, ph, pw)

        # Compute entropy
        entropy = self.compute_entropy(patches)  # (B, H_p, W_p, C)

        # Sub-patch extraction (7x7 or given dct_block_size)
        dh, dw = self.dct_block_size
        patches = patches.unfold(4, dh, 1).unfold(5, dw, 1)  # 步长1
        # patches = patches.unfold(4, dh, dh).unfold(5, dw, dw)  # (B, H_p, W_p, C, ph/dh, pw/dw, dh, dw)
        patches = patches.contiguous()
        B_p, H_p, W_p, C_p, n_dh, n_dw, dh, dw = patches.shape

        # Flatten for DCT processing
        dct_input = patches.view(-1, dh, dw)  # (dct_block_number, dct_h, dct_w)

        # Apply DCT transformation
        dct_blocks = self.dct2d(dct_input)  # (B*H_p*W_p*C*n_dh*n_dw, dh, dw)

        # Compute multidirectional variance feature
        psi_m = self.compute_direction_variance(dct_blocks)
        psi_m = psi_m.view(B, H_p, W_p, C, n_dh, n_dw).mean(
            dim=[-1, -2]
        )  # (B, H_p, W_p, C)

        # Combine features (variance * entropy)
        richness = psi_m * entropy  # (B, H_p, W_p, C)

        # print(psi_m.min(), psi_m.max(), psi_m.mean())
        # print(entropy.min(), entropy.max(), entropy.mean())

        # Return average information richness per patch
        return richness.mean(dim=-1)  # (B, H_p, W_p)

In [ ]:
class PatchGra(nn.Module):
    """
    Compute patch-wise gradient magnitude metric for information richness measurement.
    Processes input in patch format and outputs scalar values per patch.
    """
    
    def __init__(self, patch_size: Tuple[int, int] = (14, 14),):
        super().__init__()
        self.p = 1  # Original MATLAB parameter
        self.patch_size = patch_size
    
    def extract_patches(self, x: Tensor) -> Tensor:
        """Extract patches from image (without padding)."""
        B, C, H, W = x.shape
        ph, pw = self.patch_size

        # Main patch extraction (14x14 or given patch_size)
        x = x.unfold(2, ph, ph).unfold(3, pw, pw)  # (B, C, H/ph, W/pw, ph, pw)
        x = x.permute(0, 2, 3, 1, 4, 5)  # (B, H_p, W_p, C, ph, pw)

        return x.contiguous()
    
    def forward(self, x):
        """
        Args:
            x: Input tensor of shape [B, H_p, W_p, C, h, w]
               B: batch size
               H_p, W_p: number of patches in height/width dimensions
               C: channels
               h, w: spatial dimensions of each patch
               
        Returns:
            Tensor of shape [B, H_p, W_p] containing scalar values
            representing information richness per patch
        """
        x = self.extract_patches(x)  # (B, H_p, W_p, C, ph, pw)
        
        # Save original shape and reshape for patch processing
        B, H_p, W_p, C, h, w = x.shape
        x = x.contiguous().view(-1, C, h, w)  # [B*H_p*W_p, C, h, w]

        # Compute horizontal differences
        diff_h = torch.diff(x, dim=-1)  # Horizontal diff
        pad_h = (x[..., 0] - x[..., -1]).unsqueeze(-1)  # Circular padding
        u_h = torch.abs(torch.cat([diff_h, pad_h], dim=-1))

        # Compute vertical differences
        diff_v = torch.diff(x, dim=-2)  # Vertical diff
        pad_v = (x[:, :, 0, :] - x[:, :, -1, :]).unsqueeze(-2)  # Circular padding
        u_v = torch.abs(torch.cat([diff_v, pad_v], dim=-2))

        # Compute constants
        gamma = 0.5 * self.p - 1
        e = torch.tensor(torch.e, dtype=x.dtype)
        c = self.p * (e ** gamma)
        
        # Compute magnitude components
        mu_h = c * u_h - torch.pow(2 * u_h + 0.01, self.p)
        mu_v = c * u_v - torch.pow(2 * u_v + 0.01, self.p)
        
        # Combine magnitudes and reduce spatial dimensions
        S = torch.sqrt(mu_h**2 + mu_v**2)
        S_channel_mean = S.mean(dim=[-2, -1])  # Average over h,w per channel
        
        # Reshape back to original patch structure
        S_patch = S_channel_mean.view(B, H_p, W_p, C)
        
        # Final reduction: average across channels
        return S_patch.mean(dim=-1)

In [ ]:
# Load Text Image
def load_image_batch(image_paths, patch_size=14):
    batch = []
    for path in image_paths:
        img = Image.open(path).convert("RGB")
        
        H, W = img.size
        tar_h = H if H % patch_size == 0 else (H // patch_size + 1) * patch_size
        tar_w = W if W % patch_size == 0 else (W // patch_size + 1) * patch_size
        target_size = (tar_h, tar_w)
        
        img = img.resize(target_size)
        arr = np.array(img) / 255.0  # 归一化到[0,1]
        tensor = torch.tensor(arr).permute(2, 0, 1).unsqueeze(0)  # (1,3,H,W)
        batch.append(tensor)
    return torch.cat(batch, dim=0).float().to("cuda")  # (B,3,H,W)


def load_voxel_grid_batch(file_paths, norm_range=(-1, 1)):
    batch = []
    for path in file_paths:
        voxel = np.load(path)
        if norm_range is not None:
            min_val, max_val = np.min(voxel), np.max(voxel)
            voxel = (voxel - min_val) / (max_val - min_val)  # [0, 1]
            voxel = voxel * (norm_range[1] - norm_range[0]) + norm_range[0]

        tensor = torch.from_numpy(voxel).float().to("cuda")
        batch.append(tensor.unsqueeze(0))
    return torch.cat(batch, dim=0)


def visualize_voxel(voxel, save_path):
    voxel = voxel - voxel.min()
    arr = voxel[0]
    for i in range(1, voxel.shape[0]):
        arr = arr + voxel[i]
    
    cmap = matplotlib.colormaps.get_cmap("gray")
    arr = (arr - arr.min()) / (arr.max() - arr.min()) * 255.0
    arr = arr.astype(np.uint8)
    
    plt.imsave(save_path, arr, cmap=cmap)


def get_high_freq(input_batch, filtered_batch):
    high_freq_batch = input_batch - filtered_batch
    return high_freq_batch

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
info_rich_low = InformationRichnessV2().to(device)
info_rich_hight = PatchGra().to(device)

In [ ]:
out_dir = "./decouple_2"
os.makedirs(out_dir, exist_ok=True)

img1 = "/data_nvme/sph/EventScape_processed/Town05_test/Town05/sequence_0/rgb/data/05_000_0010_image.png"
img2 = "/data_nvme/sph/EventScape_processed/Town05_test/Town05/sequence_1/rgb/data/05_001_0010_image.png"
img3 = "/data_nvme/sph/EventScape_processed/Town05_test/Town05/sequence_2/rgb/data/05_002_0010_image.png"

image_paths = [img1, img2, img3]
input_batch = load_image_batch(image_paths)
print(input_batch.shape)

with torch.no_grad():
    filtered_batch = ssf_filter_batch(
        input_batch, alpha=0.8, beta=0.05, lambda_max=2e2
    )
high_freq_batch = get_high_freq(input_batch, filtered_batch)

for i in range(filtered_batch.shape[0]):
    # Save the low freq results
    output_np = filtered_batch[i].permute(1, 2, 0).cpu().numpy()
    save_p = os.path.join(out_dir, f"img_low_{i}.png")
    Image.fromarray((output_np * 255).astype(np.uint8)).save(save_p)

    high_freq_np = high_freq_batch[i].permute(1, 2, 0).cpu().numpy()

    # 方法1：线性拉伸增强对比度（适合显示细节）
    high_freq_vis = (high_freq_np * 0.5 + 0.5) * 255  # 将[-1,1]映射到[0,255]
    high_freq_vis = np.clip(high_freq_vis, 0, 255).astype(np.uint8)
    save_p = os.path.join(out_dir, f"img_high_{i}.png")
    Image.fromarray(high_freq_vis).save(save_p)

In [ ]:
img_low_richness = info_rich_low(filtered_batch)

# print(img_low_richness.min(), img_low_richness.max(), img_low_richness.mean())
# print(img_low_richness.shape)

for i in range(len(img_low_richness)):
    rich = img_low_richness[i].cpu().numpy()
    save_p = os.path.join(out_dir, f"img_low_richness_{i}.png")
    vis_depth_map(rich, save_fig=True, save_path=save_p)

img_high_richness = info_rich_hight(high_freq_batch)

# print(img_high_richness.min(), img_high_richness.max(), img_high_richness.mean())
# print(img_high_richness.shape)

for i in range(len(img_high_richness)):
    rich = img_high_richness[i].cpu().numpy()
    save_p = os.path.join(out_dir, f"img_high_richness_{i}.png")
    vis_depth_map(rich, save_fig=True, save_path=save_p)

In [ ]:
vox1 = "/data_nvme/sph/EventScape_processed/Town05_test/Town05/sequence_0/events/voxels/05_000_0010_events.npy"
vox2 = "/data_nvme/sph/EventScape_processed/Town05_test/Town05/sequence_1/events/voxels/05_001_0010_events.npy"
vox3 = "/data_nvme/sph/EventScape_processed/Town05_test/Town05/sequence_2/events/voxels/05_002_0010_events.npy"

voxel_paths = [vox1, vox2, vox3]
input_batch = load_voxel_grid_batch(voxel_paths, norm_range=(0, 1))
print("Input shape:", input_batch.shape)


with torch.no_grad():
    filtered_batch = ssf_filter_batch(
        input_batch,
        alpha=0.5,  # 降低平滑强度（事件数据通常更稀疏）
        beta=0.2,  # 放宽二阶梯度阈值
        lambda_max=5e2,
    )
# 计算高频分量
high_freq_batch = input_batch - filtered_batch

for i in range(input_batch.shape[0]):
    # 原始输入
    orig_np = input_batch[i].cpu().numpy()
    save_p = os.path.join(out_dir, f"voxel_{i}.png")
    visualize_voxel(orig_np, save_p)

    # 低频分量
    low_np = filtered_batch[i].cpu().numpy()
    save_p = os.path.join(out_dir, f"voxel_low_{i}.png")
    visualize_voxel(low_np, save_p)

    # 高频分量（增强可视化）
    high_np = high_freq_batch[i].cpu().numpy()
    # 增强对比度：乘以系数后裁剪
    high_np = np.clip(high_np * 3.0, -1.0, 1.0)  # 加强高频细节
    save_p = os.path.join(out_dir, f"voxel_high_{i}.png")
    visualize_voxel(high_np, save_p)

In [ ]:
vox_low_richness = info_rich_low(filtered_batch)

# print(vox_low_richness.min(), vox_low_richness.max(), vox_low_richness.mean())
# print(vox_low_richness.shape)

for i in range(len(vox_low_richness)):
    rich = vox_low_richness[i].cpu().numpy()
    save_p = os.path.join(out_dir, f"vox_low_richness_{i}.png")
    vis_depth_map(rich, save_fig=True, save_path=save_p)

vox_high_richness = info_rich_hight(high_freq_batch)

# print(vox_high_richness.min(), vox_high_richness.max(), vox_high_richness.mean())
# print(vox_high_richness.shape)

for i in range(len(vox_high_richness)):
    rich = vox_high_richness[i].cpu().numpy()
    save_p = os.path.join(out_dir, f"vox_high_richness_{i}.png")
    vis_depth_map(rich, save_fig=True, save_path=save_p)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class GaussianLaplacianPyramid(nn.Module):
    def __init__(self, nlev=3, scale=0.5, sigma=1.0, kernel_size=3):
        super().__init__()
        self.nlev = nlev
        self.scale = scale
        self.sigma = sigma
        self.kernel_size = kernel_size
        self.gaussian_filter = self._create_gaussian_kernel(self.kernel_size, sigma)

    def _create_gaussian_kernel(self, kernel_size, sigma):
        """创建高斯卷积核"""
        ax = torch.arange(kernel_size, dtype=torch.float32) - (kernel_size - 1) / 2.0
        xx, yy = torch.meshgrid(ax, ax, indexing="ij")
        kernel = torch.exp(-(xx**2 + yy**2) / (2 * sigma**2))
        return kernel.view(1, 1, kernel_size, kernel_size) / kernel.sum()

    def _gaussian_blur(self, x):
        """应用高斯模糊（带反射填充）"""
        pad = (
            self.gaussian_filter.shape[-1] // 2,
            self.gaussian_filter.shape[-1] // 2,
            self.gaussian_filter.shape[-2] // 2,
            self.gaussian_filter.shape[-2] // 2,
        )
        x_pad = F.pad(x, pad, mode="reflect")
        return F.conv2d(
            x_pad, self.gaussian_filter.repeat(x.size(1), 1, 1, groups=x.size(1))
        )

    def _downsample(self, x):
        """高斯模糊+下采样"""
        x_blur = self._gaussian_blur(x)
        _, _, h, w = x_blur.shape
        new_size = (int(h * self.scale), int(w * self.scale))
        return F.interpolate(x_blur, new_size, mode="bilinear", align_corners=False)

    def _upsample(self, x, target_size):
        """上采样+高斯模糊"""
        x_up = F.interpolate(x, target_size, mode="bilinear", align_corners=False)
        return self._gaussian_blur(x_up)

    def forward(self, x):
        """
        输入:
            x - 形状为(B, C, H, W)的张量

        输出:
            low_freq - 低频token (B, C, H, W)
            high_freq - 高频token (B, C, H, W)
        """
        # 构建高斯金字塔和拉普拉斯金字塔
        G = [x]
        L = []
        current = x

        for _ in range(self.nlev - 1):
            down = self._downsample(current)
            up = self._upsample(down, current.shape[-2:])
            L.append(current - up)
            G.append(down)
            current = down

        # 低频信息：最底层高斯层上采样回原尺寸
        # low_freq = self._upsample(G[-1], x.shape[-2:])
        low_freq = torch.zeros_like(x)
        for i, g in enumerate(G):
            up_g = self._upsample(g, x.shape[-2:])
            weight = 0.5 ** i  # 高层级权重衰减
            low_freq += weight * up_g

        # 高频信息：所有拉普拉斯层上采样求和
        high_freq = torch.zeros_like(x)
        for laplacian in L:
            up_lap = self._upsample(laplacian, x.shape[-2:])
            high_freq += up_lap

        return low_freq, high_freq


class SpatialFrequency(nn.Module):
    """空间频率特征提取模块"""

    def __init__(self, kernel_size=3):
        super().__init__()
        self.avg_pool = nn.AvgPool2d(kernel_size, stride=1, padding=kernel_size // 2)

    def forward(self, x):
        # 镜像填充1像素
        x_pad = F.pad(x, (1, 1, 1, 1), mode="reflect")

        # 计算水平和垂直梯度
        kernel_h = torch.tensor([0, 1, -1], dtype=torch.float32).view(1, 1, 1, 3)
        kernel_v = torch.tensor([0, 1, -1], dtype=torch.float32).view(1, 1, 3, 1)

        grad_h = F.conv2d(x_pad, kernel_h.to(x.device), padding=0)
        grad_v = F.conv2d(x_pad, kernel_v.to(x.device), padding=0)

        # 裁剪到原始尺寸
        grad_h = grad_h[:, :, 1:-1, 1:-1]
        grad_v = grad_v[:, :, 1:-1, 1:-1]

        # 计算空间频率
        rf = self.avg_pool(grad_v.pow(2))
        cf = self.avg_pool(grad_h.pow(2))
        return torch.sqrt(rf + cf)


class PyramidFeatureFusion(nn.Module):
    """多尺度特征融合模块"""

    def __init__(self, nlev=3, scale=0.5, sigma=1.0):
        super().__init__()
        self.pyramid = GaussianLaplacianPyramid(nlev, scale, sigma)
        self.sf = SpatialFrequency()

    def forward(self, x):
        # 分解为低频和高频
        low, high = self.pyramid(x)

        # 计算各层的空间频率特征
        sf_low = self.sf(low)
        sf_high = self.sf(high)

        # 特征融合
        return sf_low + sf_high

In [ ]:
import torch
import torch.nn as nn

class PatchGra(nn.Module):
    """
    Compute patch-wise gradient magnitude metric for information richness measurement.
    Processes input in patch format and outputs scalar values per patch.
    """
    
    def __init__(self):
        super().__init__()
        self.p = 0.8  # Original MATLAB parameter
    
    def forward(self, x):
        """
        Args:
            x: Input tensor of shape [B, H_p, W_p, C, h, w]
               B: batch size
               H_p, W_p: number of patches in height/width dimensions
               C: channels
               h, w: spatial dimensions of each patch
               
        Returns:
            Tensor of shape [B, H_p, W_p] containing scalar values
            representing information richness per patch
        """
        # Save original shape and reshape for patch processing
        B, H_p, W_p, C, h, w = x.shape
        x = x.contiguous().view(-1, C, h, w)  # [B*H_p*W_p, C, h, w]

        # Compute horizontal differences
        diff_h = torch.diff(x, dim=-1)  # Horizontal diff
        pad_h = (x[..., 0] - x[..., -1]).unsqueeze(-1)  # Circular padding
        u_h = torch.cat([diff_h, pad_h], dim=-1)

        # Compute vertical differences
        diff_v = torch.diff(x, dim=-2)  # Vertical diff
        pad_v = (x[:, :, 0, :] - x[:, :, -1, :]).unsqueeze(-2)  # Circular padding
        u_v = torch.cat([diff_v, pad_v], dim=-2)

        # Compute constants (MATLAB equivalent parameters)
        gamma = 0.5 * self.p - 1
        eps = torch.finfo(x.dtype).eps  # Machine epsilon for numerical stability
        c = self.p * (eps ** gamma)

        # Compute magnitude components
        mu_h = c * u_h - torch.pow(2 * u_h + 0.01, self.p)
        mu_v = c * u_v - torch.pow(2 * u_v + 0.01, self.p)

        # Combine magnitudes and reduce spatial dimensions
        S = torch.sqrt(mu_h**2 + mu_v**2)
        S_channel_mean = S.mean(dim=[-2, -1])  # Average over h,w per channel
        
        # Reshape back to original patch structure
        S_patch = S_channel_mean.view(B, H_p, W_p, C)
        
        # Final reduction: average across channels
        return S_patch.mean(dim=-1)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------
#  Channel Attention
# ----------------------
class ChannelAttention(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        # Shared MLP
        self.mlp = nn.Sequential(
            nn.Linear(channel, channel // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        
        # Avg/Max Pooling
        avg_out = self.mlp(self.avg_pool(x).view(b, c))
        max_out = self.mlp(self.max_pool(x).view(b, c))
        
        # Feature Fusion
        channel_weights = self.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        return x * channel_weights.expand_as(x)

# ----------------------
#  Spatial Attention
# ----------------------
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        assert kernel_size % 2 == 1, "Kernel size must be odd"
        
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Channel Pooling
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        combined = torch.cat([avg_out, max_out], dim=1)
        
        # Spatial Attention
        spatial_weights = self.sigmoid(self.conv(combined))
        return x * spatial_weights

# ----------------------
#  Residual Block
# ----------------------
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels)
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        x = self.conv_block(x)
        x += residual  # 残差连接
        return self.relu(x)

# ----------------------
#  Frequency Decomposition
# ----------------------
class FrequencyDecomp(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # 低频路径
        self.low_pass = nn.Sequential(
            nn.Conv2d(dim, dim//2, 5, padding=2, dilation=2),
            ChannelAttention(dim//2)
        )
        # 高频路径
        self.high_pass = nn.Sequential(
            nn.Conv2d(dim, dim//2, 3, padding=1),
            ResidualBlock(dim//2),
            SpatialAttention()
        )

    def forward(self, x):
        low = self.low_pass(x)
        high = self.high_pass(x)
        return torch.cat([low, high], dim=1)

In [ ]:
import torch
import torch.nn as nn

# ----------------------
# 轻量通道注意力 (参数减少75%)
# ----------------------
class LiteChannelAttention(nn.Module):
    def __init__(self, channel):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv2d(channel, channel, 1)  # 单层1x1卷积替代MLP
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        weights = self.sigmoid(self.conv(self.gap(x)))
        return x * weights

# ----------------------
# 轻量空间注意力 (参数减少90%)
# ----------------------
class LiteSpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(2, 1, 3, padding=1, bias=False),  # 3x3替代7x7
            nn.ReLU(inplace=True)                       # 增加非线性
        )
    
    def forward(self, x):
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        avg_out = torch.mean(x, dim=1, keepdim=True)
        combined = torch.cat([max_out, avg_out], dim=1)
        weights = self.conv(combined)
        return x * weights

# ----------------------
# 极简残差块 (参数减少60%)
# ----------------------
class LiteResidual(nn.Module):
    def __init__(self, channels):
        super().__init__()
        # 深度可分离卷积
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels),  # 深度卷积
            nn.Conv2d(channels, channels, 1),                             # 逐点卷积
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return x + self.conv(x)  # 残差连接

# ----------------------
# 轻量频率分解模块
# ----------------------
class LiteFrequencyDecomp(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # 共享初始卷积减少参数
        self.shared_conv = nn.Conv2d(dim, dim, 3, padding=1)
        
        # 低频路径
        self.low_pass = nn.Sequential(
            nn.Conv2d(dim, dim//2, 5, padding=2, dilation=2, groups=4),  # 分组卷积
            LiteChannelAttention(dim//2)
        )
        
        # 高频路径
        self.high_pass = nn.Sequential(
            LiteResidual(dim//2),
            LiteSpatialAttention()
        )

    def forward(self, x):
        x = self.shared_conv(x)          # 共享基础特征
        low = self.low_pass(x[:, ::2])   # 间隔采样降低计算量
        high = self.high_pass(x[:, 1::2])
        return torch.cat([low, high], dim=1)


In [1]:
from model.epde.decouple import FrequencyDecoupler, CrossAttention
def count_parameters(model):
    """
    计算并打印模型的参数总数
    """
    total_params = sum(p.numel() for p in model.parameters())  # 计算总参数个数
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)  # 计算可训练参数个数
    print(f"总参数个数: {total_params}")
    print(f"可训练参数个数: {trainable_params}")

In [ ]:
# a = FrequencyDecomp(dim=1024)
# count_parameters(a)
# count_parameters(a.low_pass)
# count_parameters(a.high_pass)

b = LiteFrequencyDecomp(dim=1024)
count_parameters(b)

In [ ]:
a = FrequencyDecoupler(dim=1024)
b = CrossAttention(embed_dim=1024, num_heads=8)
count_parameters(a)
count_parameters(b)

In [ ]:
class FrequencyAwareFusion(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        # 频带分解器
        self.img_decomp = FrequencyDecomp(dim)
        self.event_decomp = FrequencyDecomp(dim)
        
        # 跨频段注意力组
        self.cfa_group = nn.ModuleList([
            CrossFrequencyAttention(dim//2, num_heads) 
            for _ in range(4)  # LL/LH/HL/HH组合
        ])
        
        # 三重门控融合器
        self.tri_gate = TripleGateController(dim*2)
        
        # 频带重建器
        self.reconstructor = FrequencyFusion(dim)

    def forward(self, img, event, masks):
        """
        masks: 包含四个mask的字典
        """
        # Step1: 频带分解
        img_low, img_high = self.img_decomp(img)     # (B,C/2,H,W)*2
        eve_low, eve_high = self.event_decomp(event) # (B,C/2,H,W)*2
        
        # Step2: 四元交叉注意力
        attn_maps = []
        for i, (src, tgt) in enumerate([(img_low,eve_low), (img_low,eve_high),
                                      (img_high,eve_low), (img_high,eve_high)]):
            attn = self.cfa_group[i](src, tgt, masks[i//2]) 
            attn_maps.append(attn)
        
        # Step3: 三重门控融合
        fused = self.tri_gate(
            torch.cat([img_low, img_high], dim=1),
            torch.cat([eve_low, eve_high], dim=1),
            attn_maps,
            masks
        )
        
        # Step4: 频带重建
        out = self.reconstructor(fused)
        return out


In [ ]:
a = FrequencyDecoupler(embed_dim=1024)
count_parameters(a)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from model.epde.utils import token2feature, feature2token


class EnhancedDecoupleLoss(nn.Module):
    def __init__(self, alpha=0.7):
        super().__init__()
        self.beta = alpha

    def forward(self, origin, low, high):
        low_smooth = F.l1_loss(low, F.avg_pool2d(low, 3, stride=1, padding=1))
        high_edge = 1 / (F.conv2d(high, ...).abs().mean() + 1e-6)

        recon_loss = F.mse_loss(origin, low + high)

        return self.alpha * (low_smooth + high_edge) + recon_loss


class DCTLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, patch_grid_size):
        # The shape of x: [B, N, C]
        x_fea = token2feature(x, patch_grid_size)
        x_dct = torch.fft.fft2(x_fea)
        x_dct = feature2token(x_dct)
        return x_dct


class DualDomainFeatureDecoupler(nn.Module):
    def __init__(self, embed_dim, ratio=0.5):
        super().__init__()
        self.embed_dim = embed_dim

        # Frequency-Domain Guided Branch
        self.dct_layer = DCTLayer()
        self.freq_gate = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 4),
            nn.GELU(),
            nn.Linear(embed_dim // 4, 2),
        )

        # Spatial Refinement Branch
        self.spatial_split = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim // 4, 3, padding=1, groups=4),
            nn.GELU(),
            nn.Conv2d(embed_dim // 4, 2, 1),
        )

    def forward(self, x, patch_grid_size):
        B, N, C = x.shape

        # Frequency Domain Guidance
        x_dct = self.dct_layer(x)  # [B, N, C]
        freq_gates = self.freq_gate(x_dct.mean(dim=1))  # [B, 2]

        # Spatial domain refinement
        x_fea = token2feature(x, patch_grid_size)
        spatial_mask = self.spatial_split(x_fea)  # [B, 2, H, W]

        # Dual domain fusion
        combined_mask = F.softmax(spatial_mask * freq_gates.view(B, 2, 1, 1), dim=1)

        low = x_fea * combined_mask[:, 0:1]
        high = x_fea * combined_mask[:, 1:2]

        low = feature2token(low)
        high = feature2token(high)

        return low, high, combined_mask

In [4]:
from model.epde.decouple import DualDomainFeatureDecoupler, FeatureDecoupleFusion
# a = DualDomainFeatureDecoupler(embed_dim=1024)
# count_parameters(a)

a = FeatureDecoupleFusion(embed_dim=1024, num_heads=8)
count_parameters(a)

总参数个数: 10628104
可训练参数个数: 10628104


In [3]:
import torch
# from model.epde.decouple import DCTLayer

from torch_dct import dct_2d

x = torch.rand([3, 6, 5, 5])
freq_complex = torch.fft.fft2(x)
print(freq_complex.shape, freq_complex.dtype)

magnitude = torch.abs(freq_complex)
phase = torch.angle(freq_complex) 
magnitude_shifted = torch.fft.fftshift(magnitude, dim=(-2, -1))
print(magnitude_shifted.shape, magnitude_shifted.dtype)
# x = dct_2d(x)
# print(x.shape, x.dtype)
# a = DCTLayer()
# x = a(x, (2, 3))
# print(x.shape)
# print(x)
# print(x.dtype)
# linear = torch.nn.Linear(in_features=5, out_features=5)
# x = linear(x)

torch.Size([3, 6, 5, 5]) torch.complex64
torch.Size([3, 6, 5, 5]) torch.float32


In [ ]:
class FrequencyAwareFusion(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()

        # 跨频段注意力组
        self.cfa_group = nn.ModuleList(
            [CrossFrequencyAttention(dim // 2, num_heads) for _ in range(2)]
        )

        # 三重门控融合器
        self.tri_gate = TripleGateController(dim * 2)

    def forward(self, img, event, masks):
        """
        masks: 包含四个mask的字典
        """
        # Step1: 频带分解
        img_low, img_high = self.img_decomp(img)  # (B,C/2,H,W)*2
        eve_low, eve_high = self.event_decomp(event)  # (B,C/2,H,W)*2

        # Step2: 四元交叉注意力
        attn_maps = []
        for i, (src, tgt) in enumerate([(img_low, eve_low), (img_high, eve_high)]):
            attn = self.cfa_group[i](src, tgt, masks[i // 2])
            attn_maps.append(attn)

        # Step3: 三重门控融合
        fused = self.tri_gate(
            torch.cat([img_low, img_high], dim=1),
            torch.cat([eve_low, eve_high], dim=1),
            attn_maps,
            masks,
        )

        return fused


class CrossFrequencyAttention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.q_proj = nn.Linear(dim, dim)
        self.kv_proj = nn.Linear(dim, dim * 2)
        self.attn = nn.MultiheadAttention(dim, num_heads)

    def forward(self, src, tgt, mask):
        # 动态投影
        q = self.q_proj(src.flatten(2).permute(2, 0, 1))
        k, v = self.kv_proj(tgt.flatten(2).permute(2, 0, 1)).chunk(2, dim=-1)

        # 掩码增强注意力
        attn_out, _ = self.attn(q, k, v, key_padding_mask=(1 - mask).flatten(1).bool())
        return attn_out.permute(1, 2, 0).view_as(src)


class TripleGateController(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # 空间-通道-频段三重视觉门控
        self.spatial_gate = nn.Sequential(
            nn.Conv2d(4, 8, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(8, 2, 3, padding=1),
            nn.Sigmoid(),
        )
        self.channel_gate = nn.Linear(dim, dim * 2)
        self.freq_gate = nn.Parameter(torch.randn(2, 2))

    def forward(self, img_feat, eve_feat, attns, masks):
        # 空间门控（基于原始mask）
        spatial_weights = self.spatial_gate(
            torch.cat(
                [
                    masks["img_low"],
                    masks["img_high"],
                    masks["eve_low"],
                    masks["eve_high"],
                ],
                dim=1,
            )
        )  # (B,2,H,W)

        # 通道门控
        channel_weights = torch.sigmoid(
            self.channel_gate(img_feat.mean(dim=(2, 3)) + eve_feat.mean(dim=(2, 3)))
        ).view(
            -1, *img_feat.shape[1:]
        )  # (B,C,H,W)

        # 三阶融合
        fused = (spatial_weights[:, 0:1] * channel_weights) * img_feat + (
            spatial_weights[:, 1:2] * (1 - channel_weights)
        ) * eve_feat
        return fused

In [ ]:
class RelativeConfidenceCrossAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, reduction=1, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # Linear Projection Layers
        self.W_q = nn.Linear(embed_dim, embed_dim)  # x1 -> Query
        self.W_k = nn.Linear(embed_dim, embed_dim)  # x2 -> Key
        self.W_v = nn.Linear(embed_dim, embed_dim)  # x2 -> Value

        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x1, x2, mask_1, mask_2):
        B, N, _ = x1.shape

        # Step 1: Using mask_1 to suppress Queries with low confidence x1
        masked_x1 = x1 * mask_1.unsqueeze(-1)  # [B, N, C] * [B, N, 1]
        q = self.W_q(masked_x1)

        # Step 2: Using mask_2 to suppress Keys/Values with low confidence x2
        masked_x2 = x2 * mask_2.unsqueeze(-1)  # [B, N, C] * [B, N, 1]
        k = self.W_k(masked_x2)
        v = self.W_v(masked_x2)

        # [B, H, N, D]
        q = q.view(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        k = k.view(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        v = v.view(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        # Step 3: Calculate attention score (incorporating relative credibility)
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim**0.5)

        # [B, N, 1] - [B, 1, N] → [B, N, N]
        relative_mask = mask_1.unsqueeze(-1) - mask_2.unsqueeze(1)
        attn_scores = attn_scores + relative_mask.unsqueeze(1)  # [B, H, N, N]

        # Calculating attention weights
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Step 4: Aggregate Value
        attn_output = torch.matmul(attn_weights, v)  # [B, H, N, D]

        # Merge multiple heads and project
        attn_output = attn_output.permute(0, 2, 1, 3).contiguous().view(B, N, -1)
        attn_output = self.out_proj(attn_output)  # [B, N, C]

        return attn_output

In [ ]:
from model.epde.decouple import FeatureDecoupleFusion
a = FeatureDecoupleFusion(embed_dim=1024, num_heads=8)
count_parameters(a)
count_parameters(a.low_cross_att)

In [ ]:
a = RelativeConfidenceCrossAttention(embed_dim=1024, num_heads=8)
count_parameters(a)

In [ ]:
a = FrequencyAwareFusion(dim=1024, num_heads=8)
count_parameters(a)
count_parameters(a.tri_gate)
count_parameters(a.tri_gate.channel_gate)

数值稳定性：

最终输出使用.clamp(0,1)防止浮点误差
所有中间变量保持float32精度

### 参数选择建议
数据类型	alpha	beta	lambda_max	迭代次数

RGB图像	0.7~1.0	0.03~0.1	100~300	300~500

事件体素	0.5~0.8	0.01~0.05	500~1000	500~800

### 常见问题排查

输出全黑/全白：

检查输入是否已归一化到[0,1]

降低lambda_max或增加beta

边缘伪影：

尝试将mode='circular'改为mode='reflect'

在输入周围添加镜像填充

显存不足：

减小批处理大小

使用torch.cuda.empty_cache()

启用混合精度（需修改代码）